# 📦 Optimisation Multi-Étages de la Valeur de Stock — Modèle MILP

---

## Contexte et Problématique

Dans un contexte industriel concurrentiel, la gestion des stocks représente un enjeu financier majeur : **trop de stock immobilise du capital** et génère des risques d'obsolescence, tandis que **trop peu de stock expose à des ruptures** et dégrade le taux de service client.

Ce projet s'inscrit dans une démarche concrète d'amélioration de la performance supply chain d'un site industriel multi-étages (Matières Premières → Semi-Finis → Produits Finis). L'analyse des indicateurs de performance (KPIs) révèle un **surstockage structurel** :

- 💰 **Capital immobilisé excessif** : la valeur totale du stock dépasse significativement la cible fiscale de fin d'année.
- 📉 **Rotation insuffisante** : certaines références accumulent des niveaux bien supérieurs à leur stock de sécurité optimal.
- ⚠️ **Risque d'obsolescence** : un stock trop élevé sur des références à faible rotation peut entraîner des provisions financières.

---

## Objectif du Modèle

L'objectif est de construire un **outil d'aide à la décision** basé sur la Programmation Linéaire Mixte en Nombres Entiers (**MILP — Mixed Integer Linear Programming**) capable de répondre à trois questions opérationnelles :

| Question | Levier |
|----------|--------|
| **Quand commander ?** (MP) | Variable binaire $x_{i,t}$ |
| **Combien commander / produire ?** | Variables continues $O_{i,t}^{MP}$, $P_{j,t}^{SF}$, $P_{l,t}^{PF}$ |
| **La cible fiscale est-elle atteignable ?** | Contrainte $C5$ + statut du solveur |

---

## Architecture du Modèle

Le modèle est structuré en **deux couches** :

1. **Couche théorique — EOQ** : calcule les paramètres économiques de référence (quantité économique de commande, stock de sécurité) qui calibrent le modèle MILP.
2. **Couche décisionnelle — MILP multi-périodes, multi-étages** : optimise la trajectoire de stock de juin à septembre, sous toutes les contraintes opérationnelles réelles (lead times, temps de cycle, BOM, MOQ, MLS, fermeture fournisseur en août).

```
MP ──(achat, lead time LT_i)──► Stock MP
Stock MP ──(production, BOM a_ij)──► Stock SF    (temps de cycle ct_j)
Stock SF ──(production, BOM b_jl)──► Stock PF    (temps de cycle ct_l)
Stock PF ──(vente)──────────────────► Demande client D_{l,t}
```

---

## Principe de séparation Modèle / Données

> Ce notebook suit le principe **AMPL / GAMS** : le modèle mathématique est défini de façon **purement abstraite** (sections 1 à 4), sans aucune valeur numérique.
> Toutes les données concrètes sont regroupées **en section 5 uniquement**.
> Pour adapter le modèle à des données réelles (SAP), il suffit de modifier la section 5.


---
## 0. Imports

In [ ]:
# Pyomo : librairie de modélisation mathématique (AbstractModel, Var, Param, Constraint, Objective...)
from pyomo.environ import *
from IPython.display import display

# Pandas : manipulation de tableaux de données (affichage des résultats)
import pandas as pd

# NumPy : calculs numériques (formules EOQ, racines carrées)
import numpy as np


---
## 1. LES PARAMÈTRES

### Notations mathématiques

#### Indices (ensembles abstraits)

| Indice | Ensemble | Signification |
|--------|----------|---------------|
| $i$ | $MP$ | Références Matières Premières |
| $j$ | $SF$ | Références Semi-Finies |
| $l$ | $PF$ | Références Produits Finis |
| $t$ | $\{1, \ldots, T\}$ | Périodes mensuelles, $T$ = fin d'horizon fiscal (fin septembre) |

#### Paramètres économiques globaux

| Symbole | Description |
|---------|-------------|
| $Target$ | Cible de valeur de stock totale en fin d'horizon (\$) |
| $\tau$ | Taux de possession annuel (%) |
| $z_{\alpha}$ | Coefficient du niveau de service cible $\alpha$ (ex : $z_{0.95} = 1.65$) |
| $M$ | Grand nombre pour la linéarisation *big-M* (contraintes MOQ / MLS) |
| $V_{max}$ | Capacité de stockage globale (m³) |

#### Paramètres par référence

| Symbole | Étage | Description |
|---------|-------|-------------|
| $p_i^{MP},\, p_j^{SF},\, p_l^{PF}$ | MP / SF / PF | Valeur unitaire (\$/unité) |
| $S_{i,0}^{MP},\, S_{j,0}^{SF},\, S_{l,0}^{PF}$ | tous | Stock initial réel à $t=0$ (extrait SAP) |
| $LT_i$ | MP | Lead time fournisseur (mois) |
| $ct_j,\, ct_l$ | SF / PF | Temps de cycle de production (mois) |
| $MOQ_i$ | MP | Quantité minimale de commande |
| $MLS_j,\, MLS_l$ | SF / PF | Taille de lot minimale de production |
| $c_j,\, c_l$ | SF / PF | Charge unitaire de production (heures/unité) |
| $v_i^{MP},\, v_j^{SF},\, v_l^{PF}$ | tous | Volume unitaire de stockage (m³/unité) |
| $SS_k$ | tous | Stock de sécurité (calculé via EOQ, injecté en data) |

#### Paramètres de flux

| Symbole | Description |
|---------|-------------|
| $D_{l,t}$ | Demande client du PF $l$ en période $t$ (seul étage avec demande externe) |
| $OC_{i,t}^{MP}$ | Commandes fournisseur déjà passées, arrivant en $t$ (figées, extraites SAP) |
| $WIP_{j,t}^{SF},\, WIP_{l,t}^{PF}$ | Encours de production déjà lancés, sortant en $t$ (figés, extraits SAP) |
| $a_{ij}$ | BOM MP $\to$ SF : qté de MP $i$ nécessaire pour 1 unité de SF $j$ |
| $b_{jl}$ | BOM SF $\to$ PF : qté de SF $j$ nécessaire pour 1 unité de PF $l$ |
| $Cap_t^{SF},\, Cap_t^{PF}$ | Capacité de production disponible en $t$ (heures) |

> ⚠️ Aucune valeur numérique dans cette section — uniquement la déclaration des paramètres abstraits Pyomo.


In [ ]:
# Initialisation du modèle abstrait Pyomo
# AbstractModel : le modèle est défini sans données — elles seront injectées via DataPortal (section 5)
model = AbstractModel()

# ── Horizon temporel ─────────────────────────────────────────────────────────
model.T        = Param(within=PositiveIntegers)   # T : dernière période (fin d'horizon fiscal)
model.PER      = RangeSet(1, model.T)             # {1, 2, ..., T} : ensemble des périodes
model.t_juillet = Param(within=PositiveIntegers)  # indice de juillet (dernier mois avant fermeture)
model.AoutSet  = Set(within=model.PER)            # sous-ensemble : période(s) de fermeture fournisseur

# ── Ensembles de références par étage ────────────────────────────────────────
model.MP = Set()   # ensemble des indices Matières Premières
model.SF = Set()   # ensemble des indices Semi-Finis
model.PF = Set()   # ensemble des indices Produits Finis

# ── Paramètres économiques globaux ───────────────────────────────────────────
model.TARGET  = Param()   # cible de valeur de stock fin d'horizon ($)
model.tau     = Param()   # taux de possession annuel (ex : 0.20 = 20 %)
model.z_alpha = Param()   # coefficient niveau de service (ex : 1.65 pour alpha=95%)
model.M       = Param()   # big-M pour linéarisation des contraintes MOQ / MLS
model.V_max   = Param()   # capacité de stockage globale (m³)

# ── Paramètres par référence — Matières Premières ────────────────────────────
model.prix_MP       = Param(model.MP)   # p_i^MP : valeur unitaire ($)
model.stock_init_MP = Param(model.MP)   # S_i,0^MP : stock initial réel (t=0)
model.LT            = Param(model.MP)   # LT_i : lead time fournisseur (mois)
model.MOQ           = Param(model.MP)   # MOQ_i : quantité minimale de commande
model.volume_MP     = Param(model.MP)   # v_i^MP : volume unitaire (m³/unité)
model.SS_MP         = Param(model.MP)   # SS_i : stock de sécurité (pré-calculé EOQ)

# ── Paramètres par référence — Semi-Finis ────────────────────────────────────
model.prix_SF       = Param(model.SF)   # p_j^SF : valeur unitaire ($)
model.stock_init_SF = Param(model.SF)   # S_j,0^SF : stock initial réel (t=0)
model.ct_SF         = Param(model.SF)   # ct_j : temps de cycle production (mois)
model.MLS_SF        = Param(model.SF)   # MLS_j : taille de lot minimale
model.charge_SF     = Param(model.SF)   # c_j : charge unitaire (heures/unité)
model.volume_SF     = Param(model.SF)   # v_j^SF : volume unitaire (m³/unité)
model.SS_SF         = Param(model.SF)   # SS_j : stock de sécurité (pré-calculé EOQ)

# ── Paramètres par référence — Produits Finis ────────────────────────────────
model.prix_PF       = Param(model.PF)   # p_l^PF : valeur unitaire ($)
model.stock_init_PF = Param(model.PF)   # S_l,0^PF : stock initial réel (t=0)
model.ct_PF         = Param(model.PF)   # ct_l : temps de cycle production (mois)
model.MLS_PF        = Param(model.PF)   # MLS_l : taille de lot minimale
model.charge_PF     = Param(model.PF)   # c_l : charge unitaire (heures/unité)
model.volume_PF     = Param(model.PF)   # v_l^PF : volume unitaire (m³/unité)
model.SS_PF         = Param(model.PF)   # SS_l : stock de sécurité (pré-calculé EOQ)

# ── Demande client (uniquement sur les Produits Finis) ───────────────────────
# D_{l,t} : seul l'étage PF a une demande externe réelle (commandes clients)
model.demande = Param(model.PF, model.PER)

# ── Pipeline déjà engagé (données figées, extraites de SAP) ──────────────────
# Ces paramètres représentent des flux DÉJÀ DÉCIDÉS avant le démarrage du modèle.
# Le modèle ne peut pas les modifier — ce sont des constantes.
model.OC     = Param(model.MP, model.PER)   # OC_{i,t} : commandes MP en transit
model.WIP_SF = Param(model.SF, model.PER)   # WIP_{j,t}^SF : encours production SF
model.WIP_PF = Param(model.PF, model.PER)   # WIP_{l,t}^PF : encours production PF

# ── Nomenclature BOM (Bill of Materials) ─────────────────────────────────────
# default=0 : si un couple (i,j) ou (j,l) n'est pas déclaré en data, la valeur est 0
model.a = Param(model.MP, model.SF, default=0)   # a_{ij} : MP -> SF
model.b = Param(model.SF, model.PF, default=0)   # b_{jl} : SF -> PF

# ── Capacités de production par période ──────────────────────────────────────
# Cap_t^SF / Cap_t^PF : exprimées en heures disponibles par mois
# Réduite en août (congés fournisseurs et production)
model.Cap_SF = Param(model.PER)   # capacité atelier SF (heures/mois)
model.Cap_PF = Param(model.PER)   # capacité atelier PF (heures/mois)


---
## 2. LES VARIABLES DE DÉCISION

### Formulation mathématique

Le modèle distingue deux conventions temporelles importantes :

- Les **stocks** $S_{k,t}$ sont des **photos de fin de période** $t$ (après tous les mouvements du mois).
- Les **décisions** ($O$, $P$, $x$, $y$) sont **lancées en début de période** $t$ et produisent leur effet après un délai ($LT_i$ ou $ct_k$).

| Variable | Domaine | Convention | Signification |
|----------|---------|------------|---------------|
| $S_{i,t}^{MP}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock MP $i$ en fin de période $t$ |
| $S_{j,t}^{SF}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock SF $j$ en fin de période $t$ |
| $S_{l,t}^{PF}$ | $\mathbb{R}^+$ | Fin de $t$ | Stock PF $l$ en fin de période $t$ |
| $O_{i,t}^{MP}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **commandée** (achat MP $i$) lancée en début de $t$ |
| $P_{j,t}^{SF}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **lancée en production** SF $j$ en début de $t$ |
| $P_{l,t}^{PF}$ | $\mathbb{R}^+$ | Début de $t$ | Quantité **lancée en production** PF $l$ en début de $t$ |
| $x_{i,t}$ | $\{0, 1\}$ | Début de $t$ | **1** si une commande est passée pour MP $i$ en $t$, **0** sinon |
| $y_{j,t}^{SF}$ | $\{0, 1\}$ | Début de $t$ | **1** si un OF est lancé pour SF $j$ en $t$, **0** sinon |
| $y_{l,t}^{PF}$ | $\{0, 1\}$ | Début de $t$ | **1** si un OF est lancé pour PF $l$ en $t$, **0** sinon |

**Note sur le décalage temporel :**
Ce qui est commandé/lancé en début de $t$ n'arrive en stock qu'en fin de $t + LT_i$ (ou $t + ct_k$).
C'est ce décalage qui est encodé dans les contraintes de bilan **C1 / C2 / C3**.


In [ ]:
# ── Stocks en fin de période t ────────────────────────────────────────────────
# domain=NonNegativeReals impose S >= 0 (contrainte C13 implicite)
model.S_MP = Var(model.MP, model.PER, domain=NonNegativeReals)  # stock MP fin de t
model.S_SF = Var(model.SF, model.PER, domain=NonNegativeReals)  # stock SF fin de t
model.S_PF = Var(model.PF, model.PER, domain=NonNegativeReals)  # stock PF fin de t

# ── Quantités commandées (MP) et lancées en production (SF, PF) ───────────────
# Lancées en début de t — l'effet en stock est décalé par LT_i ou ct_k (voir C1/C2/C3)
model.O_MP = Var(model.MP, model.PER, domain=NonNegativeReals)  # O_{i,t}^MP
model.P_SF = Var(model.SF, model.PER, domain=NonNegativeReals)  # P_{j,t}^SF
model.P_PF = Var(model.PF, model.PER, domain=NonNegativeReals)  # P_{l,t}^PF

# ── Variables binaires de déclenchement ──────────────────────────────────────
# domain=Binary impose x, y in {0, 1} (contrainte C13 implicite)
model.x    = Var(model.MP, model.PER, domain=Binary)   # x_{i,t} : commande MP déclenchée ?
model.y_SF = Var(model.SF, model.PER, domain=Binary)   # y_{j,t} : OF SF déclenché ?
model.y_PF = Var(model.PF, model.PER, domain=Binary)   # y_{l,t} : OF PF déclenché ?


---
## 3. LA FONCTION OBJECTIF

### Formulation mathématique

$$\min\; Z = \sum_{t=1}^{T}\left(
  \sum_{i \in MP} p_i^{MP}\, S_{i,t}^{MP}
+ \sum_{j \in SF} p_j^{SF}\, S_{j,t}^{SF}
+ \sum_{l \in PF} p_l^{PF}\, S_{l,t}^{PF}
\right)$$

**Interprétation :**
On minimise la **valeur financière totale du stock immobilisé**, cumulée sur tout l'horizon (juin → septembre).

- Multiplier chaque quantité en stock par son prix unitaire donne la **valeur immobilisée** à cette période.
- Sommer sur toutes les périodes (et pas seulement la dernière) évite un déstockage artificiel concentré en fin d'horizon : le modèle est incité à réduire le stock **dès le début**, pas seulement en septembre.
- La hiérarchie des prix $p^{PF} > p^{SF} > p^{MP}$ fait que le modèle **priorise naturellement la réduction des stocks de produits finis**, qui immobilisent le plus de valeur ajoutée.


In [ ]:
# ── Fonction objectif ─────────────────────────────────────────────────────────
# Minimiser la somme des valeurs de stock (prix * quantité) sur tous les étages et toutes les périodes
def cost_rule(m):
    return (
        # Valeur immobilisée en Matières Premières
        sum(m.prix_MP[i] * m.S_MP[i, t] for i in m.MP for t in m.PER)
        # Valeur immobilisée en Semi-Finis
      + sum(m.prix_SF[j] * m.S_SF[j, t] for j in m.SF for t in m.PER)
        # Valeur immobilisée en Produits Finis
      + sum(m.prix_PF[l] * m.S_PF[l, t] for l in m.PF for t in m.PER)
    )

# sense=minimize : on minimise (par opposition à maximize)
model.cost = Objective(rule=cost_rule, sense=minimize)


---
## 3 (suite). LES CONTRAINTES C1 → C13

---

### C1 — Bilan de stock Matière Première

$$S_{i,t}^{MP} = \underbrace{S_{i,t-1}^{MP}}_{\text{stock précédent}}
+ \underbrace{OC_{i,t}^{MP}}_{\substack{\text{pipeline SAP}\\\text{(figé)}}}
+ \underbrace{O_{i,\;t-LT_i}^{MP}}_{\substack{\text{décision modèle}\\\text{(décalée de }LT_i\text{)}}}
- \underbrace{\sum_{j \in SF} a_{ij}\, P_{j,t}^{SF}}_{\text{consommation par la production SF}}
\quad \forall\, i \in MP,\; t \in T$$

**Points clés :**
- Si $t = 1$ : $S_{i,t-1}^{MP} = S_{i,0}^{MP}$ (stock initial réel SAP) → **C4 intégrée ici**.
- Si $t - LT_i < 1$ : la commande décidée par le modèle n'est pas encore arrivée → terme nul.
- La consommation de MP a lieu **au moment du lancement** de la production SF (pas à sa sortie).

---

### C2 — Bilan de stock Semi-Fini

$$S_{j,t}^{SF} = S_{j,t-1}^{SF}
+ \underbrace{WIP_{j,t}^{SF}}_{\substack{\text{encours déjà}\\\text{lancé (SAP)}}}
+ \underbrace{P_{j,\;t-ct_j}^{SF}}_{\substack{\text{décision modèle}\\\text{(décalée de }ct_j\text{)}}}
- \underbrace{\sum_{l \in PF} b_{jl}\, P_{l,t}^{PF}}_{\text{consommation par la production PF}}
\quad \forall\, j \in SF,\; t \in T$$

---

### C3 — Bilan de stock Produit Fini

$$S_{l,t}^{PF} = S_{l,t-1}^{PF}
+ \underbrace{WIP_{l,t}^{PF}}_{\text{encours (SAP)}}
+ \underbrace{P_{l,\;t-ct_l}^{PF}}_{\text{décision modèle}}
- \underbrace{D_{l,t}}_{\substack{\text{demande client}\\\text{(seul étage avec}\\\text{demande externe)}}}
\quad \forall\, l \in PF,\; t \in T$$

---

### C4 — Initialisation des stocks

$$S_{i,0}^{MP} = \text{stock\_init\_MP}[i], \quad
S_{j,0}^{SF} = \text{stock\_init\_SF}[j], \quad
S_{l,0}^{PF} = \text{stock\_init\_PF}[l]$$

> **Intégrée dans C1/C2/C3** via la condition $t = 1$ — pas de contrainte Pyomo séparée.

---

### C5 — Cible de valeur de stock fin d'année fiscale

$$\sum_{i \in MP} p_i^{MP}\, S_{i,T}^{MP}
+ \sum_{j \in SF} p_j^{SF}\, S_{j,T}^{SF}
+ \sum_{l \in PF} p_l^{PF}\, S_{l,T}^{PF}
\leq Target$$

**Contrainte principale du modèle** : la valeur totale du stock en fin d'horizon (fin septembre) doit être inférieure ou égale à la cible fiscale $Target$.

---

### C6 — Taux de service minimum (Produit Fini uniquement)

$$S_{l,t}^{PF} \geq SS_l \quad \forall\, l \in PF,\; t \in T$$

Le stock de PF ne doit jamais descendre en dessous du stock de sécurité $SS_l$ (calculé via EOQ), garantissant un niveau de service $\alpha$ vis-à-vis des clients.

---

### C7 — Capacité de production

$$\underbrace{\sum_{j \in SF} c_j\, P_{j,t}^{SF}}_{\text{charge totale générée}} \leq Cap_t^{SF} \quad \forall\, t$$

$$\underbrace{\sum_{l \in PF} c_l\, P_{l,t}^{PF}}_{\text{charge totale générée}} \leq Cap_t^{PF} \quad \forall\, t$$

- $c_k$ = **charge unitaire** (heures nécessaires pour produire 1 unité) — paramètre par référence.
- $Cap_t$ = **capacité disponible** (heures totales du mois) — réduite en août pour les congés.

---

### C8 — Capacité de stockage globale (volume physique)

$$\sum_{i \in MP} v_i^{MP}\, S_{i,t}^{MP}
+ \sum_{j \in SF} v_j^{SF}\, S_{j,t}^{SF}
+ \sum_{l \in PF} v_l^{PF}\, S_{l,t}^{PF}
\leq V_{max} \quad \forall\, t$$

Le volume total occupé ne doit pas dépasser la capacité physique de l'entrepôt.

---

### C9 — MOQ et liaison binaire — Achat Matière Première

$$O_{i,t}^{MP} \geq MOQ_i \cdot x_{i,t} \quad \forall\, i, t \qquad \text{(quantité min si commande)}$$
$$O_{i,t}^{MP} \leq M \cdot x_{i,t} \quad \forall\, i, t \qquad \text{(force zéro si pas de commande)}$$

- Si $x_{i,t} = 0$ → $O_{i,t}^{MP} = 0$ (aucune commande)
- Si $x_{i,t} = 1$ → $O_{i,t}^{MP} \geq MOQ_i$ (commande ≥ quantité minimale)

---

### C10 — MLS et liaison binaire — Production SF et PF

$$P_{j,t}^{SF} \geq MLS_j \cdot y_{j,t}^{SF}, \quad P_{j,t}^{SF} \leq M \cdot y_{j,t}^{SF} \quad \forall\, j, t$$
$$P_{l,t}^{PF} \geq MLS_l \cdot y_{l,t}^{PF}, \quad P_{l,t}^{PF} \leq M \cdot y_{l,t}^{PF} \quad \forall\, l, t$$

Même logique que C9 : si un OF est lancé ($y = 1$), la quantité produite doit respecter la taille de lot minimale $MLS$.

---

### C11 — Fermeture fournisseur en août

$$O_{i,t}^{MP} = 0 \quad \forall\, i \in MP,\; t \in t_{ao\hat{u}t}$$

Les fournisseurs sont fermés en août : aucune nouvelle commande MP ne peut être passée.

---

### C12 — Couverture de stock avant fermeture

$$S_{i,\, t_{juillet}}^{MP} \geq \sum_{t \in t_{ao\hat{u}t}} \sum_{j \in SF} a_{ij}\, P_{j,t}^{SF} \quad \forall\, i \in MP$$

Le stock MP disponible fin juillet doit couvrir **toute la consommation prévue en août**, puisqu'aucun réapprovisionnement ne sera possible.
Cette contrainte force le modèle à constituer un **stock tampon en juillet** → pic visible sur la trajectoire.

---

### C13 — Non-négativité et domaine des variables

$$S^{MP},\, S^{SF},\, S^{PF},\, O^{MP},\, P^{SF},\, P^{PF} \geq 0 \qquad x_{i,t},\, y_{j,t}^{SF},\, y_{l,t}^{PF} \in \{0, 1\}$$

> **Intégrée dans la déclaration des variables** via `domain=NonNegativeReals` et `domain=Binary` — pas de contrainte Pyomo séparée.


In [ ]:
# ── C1 — Bilan de stock Matière Première ─────────────────────────────────────
# S[i,t] = S[i,t-1] + OC[i,t] + O[i, t-LT_i] - sum_j(a[i,j] * P_SF[j,t])
def c1_rule(m, i, t):
    # Stock de la période précédente (ou stock initial si t=1 → intègre C4)
    S_prev = m.stock_init_MP[i] if t == 1 else m.S_MP[i, t-1]

    # Réception issue d'une commande DÉCIDÉE par le modèle, lancée t-LT_i périodes avant
    t_cmd     = t - m.LT[i]
    reception = m.O_MP[i, t_cmd] if t_cmd >= 1 else 0  # nul si hors horizon

    # Consommation de MP i par toutes les productions SF lancées en t (via BOM)
    conso = sum(m.a[i, j] * m.P_SF[j, t] for j in m.SF if m.a[i, j] > 0)

    return m.S_MP[i, t] == S_prev + m.OC[i, t] + reception - conso

model.C1_Bilan_MP = Constraint(model.MP, model.PER, rule=c1_rule)

# ── C2 — Bilan de stock Semi-Fini ─────────────────────────────────────────────
# S[j,t] = S[j,t-1] + WIP_SF[j,t] + P_SF[j, t-ct_j] - sum_l(b[j,l] * P_PF[l,t])
def c2_rule(m, j, t):
    S_prev = m.stock_init_SF[j] if t == 1 else m.S_SF[j, t-1]

    # Sortie de production SF décidée par le modèle, lancée t-ct_j périodes avant
    t_lanc    = t - m.ct_SF[j]
    reception = m.P_SF[j, t_lanc] if t_lanc >= 1 else 0

    # Consommation de SF j par toutes les productions PF lancées en t (via BOM)
    conso = sum(m.b[j, l] * m.P_PF[l, t] for l in m.PF if m.b[j, l] > 0)

    return m.S_SF[j, t] == S_prev + m.WIP_SF[j, t] + reception - conso

model.C2_Bilan_SF = Constraint(model.SF, model.PER, rule=c2_rule)

# ── C3 — Bilan de stock Produit Fini ──────────────────────────────────────────
# S[l,t] = S[l,t-1] + WIP_PF[l,t] + P_PF[l, t-ct_l] - D[l,t]
def c3_rule(m, l, t):
    S_prev = m.stock_init_PF[l] if t == 1 else m.S_PF[l, t-1]

    # Sortie de production PF décidée par le modèle, lancée t-ct_l périodes avant
    t_lanc    = t - m.ct_PF[l]
    reception = m.P_PF[l, t_lanc] if t_lanc >= 1 else 0

    # Demande client externe : seul flux de sortie non contrôlable par le modèle
    return m.S_PF[l, t] == S_prev + m.WIP_PF[l, t] + reception - m.demande[l, t]

model.C3_Bilan_PF = Constraint(model.PF, model.PER, rule=c3_rule)

# C4 — Initialisation : intégrée dans C1/C2/C3 via la condition t==1 (pas de contrainte séparée)

# ── C5 — Cible de valeur de stock fin d'année fiscale ─────────────────────────
# sum_i(p_i * S_MP[i,T]) + sum_j(p_j * S_SF[j,T]) + sum_l(p_l * S_PF[l,T]) <= TARGET
def c5_rule(m):
    return (
        sum(m.prix_MP[i] * m.S_MP[i, m.T] for i in m.MP)
      + sum(m.prix_SF[j] * m.S_SF[j, m.T] for j in m.SF)
      + sum(m.prix_PF[l] * m.S_PF[l, m.T] for l in m.PF)
      <= m.TARGET
    )
model.C5_Cible_Fiscale = Constraint(rule=c5_rule)

# ── C6 — Taux de service minimum (Produits Finis uniquement) ──────────────────
# S_PF[l,t] >= SS_l pour tout l, t
def c6_rule(m, l, t):
    return m.S_PF[l, t] >= m.SS_PF[l]   # SS_PF[l] calculé via EOQ, injecté en data
model.C6_TauxService = Constraint(model.PF, model.PER, rule=c6_rule)

# ── C7 — Capacité de production (charge totale <= capacité disponible) ─────────
# sum_j(c_j * P_SF[j,t]) <= Cap_SF[t]   pour tout t
def c7_sf_rule(m, t):
    return sum(m.charge_SF[j] * m.P_SF[j, t] for j in m.SF) <= m.Cap_SF[t]
model.C7_Capacite_SF = Constraint(model.PER, rule=c7_sf_rule)

# sum_l(c_l * P_PF[l,t]) <= Cap_PF[t]   pour tout t
def c7_pf_rule(m, t):
    return sum(m.charge_PF[l] * m.P_PF[l, t] for l in m.PF) <= m.Cap_PF[t]
model.C7_Capacite_PF = Constraint(model.PER, rule=c7_pf_rule)

# ── C8 — Capacité de stockage globale (volume physique) ───────────────────────
# sum(v_i*S_MP + v_j*S_SF + v_l*S_PF) <= V_max   pour tout t
def c8_rule(m, t):
    return (
        sum(m.volume_MP[i] * m.S_MP[i, t] for i in m.MP)
      + sum(m.volume_SF[j] * m.S_SF[j, t] for j in m.SF)
      + sum(m.volume_PF[l] * m.S_PF[l, t] for l in m.PF)
      <= m.V_max
    )
model.C8_Capacite_Stockage = Constraint(model.PER, rule=c8_rule)

# ── C9 — MOQ et liaison binaire — Achat MP ────────────────────────────────────
# O_MP[i,t] >= MOQ_i * x[i,t]   : si commande, quantité >= MOQ
def c9a_rule(m, i, t):
    return m.O_MP[i, t] >= m.MOQ[i] * m.x[i, t]
model.C9a_MOQ_min = Constraint(model.MP, model.PER, rule=c9a_rule)

# O_MP[i,t] <= M * x[i,t]   : si pas de commande (x=0), force O_MP=0
def c9b_rule(m, i, t):
    return m.O_MP[i, t] <= m.M * m.x[i, t]
model.C9b_MOQ_max = Constraint(model.MP, model.PER, rule=c9b_rule)

# ── C10 — MLS et liaison binaire — Production SF / PF ─────────────────────────
# Même logique que C9 mais pour les lancements de production

# P_SF[j,t] >= MLS_j * y_SF[j,t]
def c10a_sf_rule(m, j, t):
    return m.P_SF[j, t] >= m.MLS_SF[j] * m.y_SF[j, t]
model.C10a_MLS_SF_min = Constraint(model.SF, model.PER, rule=c10a_sf_rule)

# P_SF[j,t] <= M * y_SF[j,t]
def c10b_sf_rule(m, j, t):
    return m.P_SF[j, t] <= m.M * m.y_SF[j, t]
model.C10b_MLS_SF_max = Constraint(model.SF, model.PER, rule=c10b_sf_rule)

# P_PF[l,t] >= MLS_l * y_PF[l,t]
def c10a_pf_rule(m, l, t):
    return m.P_PF[l, t] >= m.MLS_PF[l] * m.y_PF[l, t]
model.C10a_MLS_PF_min = Constraint(model.PF, model.PER, rule=c10a_pf_rule)

# P_PF[l,t] <= M * y_PF[l,t]
def c10b_pf_rule(m, l, t):
    return m.P_PF[l, t] <= m.M * m.y_PF[l, t]
model.C10b_MLS_PF_max = Constraint(model.PF, model.PER, rule=c10b_pf_rule)

# ── C11 — Fermeture fournisseur en août (aucune commande possible) ─────────────
# O_MP[i,t] = 0  pour tout i, t dans AoutSet
def c11_rule(m, i, t):
    return m.O_MP[i, t] == 0
model.C11_Fermeture_Fournisseur = Constraint(model.MP, model.AoutSet, rule=c11_rule)

# ── C12 — Couverture de stock avant fermeture ──────────────────────────────────
# S_MP[i, t_juillet] >= sum_{t in AoutSet} sum_j(a[i,j] * P_SF[j,t])
# Stock fin juillet >= toute la consommation MP prévue en août
def c12_rule(m, i):
    conso_aout = sum(
        m.a[i, j] * m.P_SF[j, t]
        for t in m.AoutSet
        for j in m.SF
        if m.a[i, j] > 0
    )
    return m.S_MP[i, m.t_juillet] >= conso_aout
model.C12_Couverture_Aout = Constraint(model.MP, rule=c12_rule)

# C13 — Non-négativité et domaine binaire :
#        déjà intégrés via domain=NonNegativeReals et domain=Binary dans la déclaration des variables


---
## 4. RÉSOLUTION (fonction générique)

Cette fonction encapsule l'appel au solveur.
Elle sera appelée dans la section 5 (données), après la construction de l'instance concrète.


In [ ]:
def resoudre(instance, solveur="cbc", verbose=True):
    """
    Résout l'instance concrète du modèle MILP.

    Paramètres
    ----------
    instance : instance Pyomo concrète (créée via model.create_instance(data))
    solveur  : nom du solveur à utiliser (défaut : 'cbc', open-source)
    verbose  : afficher les logs du solveur (défaut : True)

    Retourne
    --------
    resultats : objet Pyomo contenant le statut et les informations de résolution
    """
    opt       = SolverFactory(solveur)   # initialise le solveur
    resultats = opt.solve(instance, tee=verbose)
    return resultats


---
## 5. GÉNÉRATION LOGIQUE DES DONNÉES MANQUANTES

Ces paramètres ne sont pas disponibles dans mes fichiers sources (volumes de stockage, charges de production, capacités, coûts fixes). Je les génère ici de façon **déterministe et reproductible**, à partir de règles industrielles calibrées sur le secteur connectique / assemblage électronique (TE Connectivity).

> Ces fonctions sont définies ici, avant toute lecture de fichier. Elles sont appelées automatiquement dans la section 6 une fois les listes MP / SF / PF connues.

---

### 5.1 — Volumes unitaires de stockage $v_k$ (m³/unité)

**Règle :** le volume est proportionnel à la complexité estimée par le prix unitaire. Les composants MP sont les plus petits, les PF emballés sont les plus grands.

$$v_i^{MP} = 0.003 \times \left(\frac{p_i}{50}\right)^{0.4} \qquad v_j^{SF} = 0.025 \times \left(\frac{p_j}{200}\right)^{0.5} \qquad v_l^{PF} = 0.07 \times \left(\frac{p_l}{800}\right)^{0.5}$$


In [ ]:
import math

def generer_volumes(prix_MP, prix_SF, prix_PF):
    """
    Génère les volumes unitaires de stockage (m³/unité) pour chaque référence.
    Règle : proportionnel à la complexité estimée par le prix unitaire.
    MP (composants)  : 0.001 – 0.050 m³    |  formule : 0.003 × (p/50)^0.4
    SF (semi-finis)  : 0.015 – 0.100 m³    |  formule : 0.025 × (p/200)^0.5
    PF (finis embal) : 0.050 – 0.200 m³    |  formule : 0.070 × (p/800)^0.5
    """
    volume_MP = {i: round(0.003 * (p / 50)  ** 0.4, 4) for i, p in prix_MP.items()}
    volume_SF = {j: round(0.025 * (p / 200) ** 0.5, 4) for j, p in prix_SF.items()}
    volume_PF = {l: round(0.070 * (p / 800) ** 0.5, 4) for l, p in prix_PF.items()}
    return volume_MP, volume_SF, volume_PF

print("✅  Fonction generer_volumes() définie")


---
### 5.2 — Charges de production $c_k$ (heures/unité) et temps de cycle $ct_k$ (mois)

**Règle :** la charge de production augmente avec la complexité du produit (proxy = prix unitaire). Les MP ne sont pas produites en interne (charge = 0).

$$c_j^{SF} = 1.0 + \frac{p_j}{300} \qquad c_l^{PF} = 2.0 + \frac{p_l}{400}$$

**Temps de cycle :** SF avec charge $\geq 3$ h → 2 mois, sinon 1 mois. PF : toujours 1 mois (assemblage final).


In [ ]:
def generer_charges_et_cycles(prix_SF, prix_PF):
    """
    Génère les charges unitaires (h/unité) et les temps de cycle (mois).
    SF : charge = 1.0 + prix/300   |  ct = 2 mois si charge >= 3h, sinon 1 mois
    PF : charge = 2.0 + prix/400   |  ct = 1 mois (assemblage final rapide)
    """
    charge_SF = {j: round(1.0 + p / 300, 2) for j, p in prix_SF.items()}
    charge_PF = {l: round(2.0 + p / 400, 2) for l, p in prix_PF.items()}
    ct_SF     = {j: 2 if charge_SF[j] >= 3.0 else 1 for j in prix_SF}
    ct_PF     = {l: 1 for l in prix_PF}   # assemblage final = 1 mois systématiquement
    return charge_SF, charge_PF, ct_SF, ct_PF

print("✅  Fonction generer_charges_et_cycles() définie")


---
### 5.3 — Capacités de production $Cap_t^{SF}$, $Cap_t^{PF}$ (heures/mois)

**Base industrielle TE Connectivity :**
- Atelier SF : 2 lignes d'assemblage × 8 h × 22 jours × 85 % OEE → **300 h/mois**
- Atelier PF : 1 ligne d'assemblage final × 8 h × 22 jours × 85 % OEE → **150 h/mois**
- **Août** : fermeture estivale standard → 35 % de la capacité normale


In [ ]:
def generer_capacites(T, t_aout):
    """
    Génère les capacités de production par période (heures/mois).
    Base SF : 2 lignes × 8h × 22j × 85% OEE = 300 h/mois
    Base PF : 1 ligne  × 8h × 22j × 85% OEE = 150 h/mois
    Août    : × 35% (fermeture estivale)
    """
    base_SF = round(2 * 8 * 22 * 0.85)   # 300 h
    base_PF = round(1 * 8 * 22 * 0.85)   # 150 h
    f_aout  = 0.35                        # 35 % capacité résiduelle en août

    Cap_SF = {t: round(base_SF * f_aout) if t in t_aout else base_SF for t in T}
    Cap_PF = {t: round(base_PF * f_aout) if t in t_aout else base_PF for t in T}
    return Cap_SF, Cap_PF

print("✅  Fonction generer_capacites() définie")


---
### 5.4 — Coûts fixes de passation / lancement (pour calibration EOQ)

**Règle :** le coût fixe de passation (MP) ou de lancement d'un OF (SF/PF) est proportionnel au prix unitaire — les références chères justifient plus d'effort administratif.

$$C_i^{MP} = \max(100, 2.5 \times p_i) \qquad C_j^{SF} = \max(150, 1.5 \times p_j) \qquad C_l^{PF} = \max(200, p_l)$$


In [ ]:
def generer_couts_fixes(prix_MP, prix_SF, prix_PF):
    """
    Génère les coûts fixes de passation de commande (MP)
    et de lancement d'un OF (SF, PF) utilisés dans la formule EOQ.
    """
    C_passation = {i: max(100, round(p * 2.5)) for i, p in prix_MP.items()}
    C_lancement_SF = {j: max(150, round(p * 1.5)) for j, p in prix_SF.items()}
    C_lancement_PF = {l: max(200, round(p * 1.0)) for l, p in prix_PF.items()}
    return C_passation, C_lancement_SF, C_lancement_PF

print("✅  Fonction generer_couts_fixes() définie")


---
### 5.5 — Écart-type de la demande $\sigma_{D_k}$ (pour calcul du stock de sécurité)

**Règle :**
- PF : 15 % de la demande mensuelle moyenne (variabilité client)
- SF : 12 % du stock initial (proxy de la variabilité de production)
- MP : 8 % du stock initial (moins volatile, lissé par les commandes groupées)


In [ ]:
def generer_sigma_D(stock_init_MP, stock_init_SF, stock_init_PF, demande, T):
    """
    Génère les écart-types de demande (sigma_D) par référence.
    PF : 15% de la demande mensuelle moyenne  (variabilité demande client)
    SF :  12% du stock initial                (variabilité de consommation interne)
    MP :   8% du stock initial                (variabilité des approvisionnements)
    """
    sigma = {}
    for l in stock_init_PF:
        dem_moy = sum(demande[(l, t)] for t in T) / len(T)
        sigma[l] = max(5, round(dem_moy * 0.15))
    for j in stock_init_SF:
        sigma[j] = max(5, round(stock_init_SF[j] * 0.12))
    for i in stock_init_MP:
        sigma[i] = max(5, round(stock_init_MP[i] * 0.08))
    return sigma

print("✅  Fonction generer_sigma_D() définie")


---
## 6. MISE EN PLACE DE LA DATA

Toutes les données du modèle sont chargées ici depuis les fichiers Excel. L'ordre de lecture est important : le fichier **Stock** est lu en premier car il définit les listes de références MP / SF / PF dont tous les autres fichiers ont besoin.

| # | Fichier | Ce qu'il fournit |
|---|---------|-----------------|
| 6.1 | `Stock.xlsx` | Listes I/J/L, prix unitaires, stocks initiaux |
| 6.2 | `Parametres_Modele.xlsx` | TARGET, V\_max, tau, z\_alpha, M |
| 6.3 | `LeadTime.xlsx` | LT fournisseur (jours → mois) |
| 6.4 | `MOQ_MLS.xlsx` | MOQ pour MP, MLS pour SF/PF |
| 6.5 | `Demande.xlsx` | Demande hebdomadaire → agrégation mensuelle |
| 6.6 | `WIP_Achats_En_Cours.xlsx` | Pipeline SAP (OC, WIP SF, WIP PF) |
| 6.7 | `BOM_SAP.xlsx` | Nomenclature a[i][j] et b[j][l] |
| 6.8 | Génération | Volumes, charges, capacités, σ\_D, coûts fixes |
| 6.9 | Calcul EOQ | Stocks de sécurité SS\_k |
| 6.10 | DataPortal | Création de l'instance Pyomo |


---
### 6.0 — Dossier de données

> Seule variable à modifier : `DOSSIER_DATA`

In [ ]:
import os, math
import pandas as pd
import numpy as np
from pyomo.environ import DataPortal, value

# ══════════════════════════════════════════════════════
# ▶  SEUL PARAMÈTRE À ADAPTER  ◀
DOSSIER_DATA = r"."    # ex: r"C:/Users/Ton_Nom/Projet/Data"
# ══════════════════════════════════════════════════════

def f(nom): return os.path.join(DOSSIER_DATA, nom)

# Horizon temporel
T             = [1, 2, 3, 4]
T_max         = 4
t_juillet     = 2
t_aout        = [3]
noms_periodes = {1: "Juin", 2: "Juillet", 3: "Août", 4: "Septembre"}

# Correspondance semaine FY → index t du modèle
# FY25-W36..W39 = Juin | W40..W43 = Juillet | W44..W47 = Août | W48..W51 = Septembre
week_to_t = {}
for w in range(36, 40): week_to_t[f"FY25-W{w:02d}"] = 1
for w in range(40, 44): week_to_t[f"FY25-W{w:02d}"] = 2
for w in range(44, 48): week_to_t[f"FY25-W{w:02d}"] = 3
for w in range(48, 52): week_to_t[f"FY25-W{w:02d}"] = 4

print(f"✅  Dossier data  : {os.path.abspath(DOSSIER_DATA)}")
print(f"   Périodes      : {[noms_periodes[t] for t in T]}")


---
### 6.1 — Référentiel produits (`Stock.xlsx`)

Ce fichier contient une ligne par PN avec les colonnes :
- `Material` : code PN
- `Material Type` : `ZROH` = MP, `ZHLB` = SF, `ZFRT` = PF
- `Stock Actuel (unités)` : quantité en stock
- `Prix Unitaire ($)` : valeur unitaire standard

La colonne `Material Type` détermine les listes $I$ (MP), $J$ (SF), $L$ (PF) qui pilotent tout le reste.


In [ ]:
df_stock = pd.read_excel(f("Stock.xlsx"), sheet_name="Stock_Actuel")
df_stock.columns = df_stock.columns.str.strip()

# Supprime la ligne TOTAL si elle est présente
df_stock = df_stock[df_stock["Material"].notna()].copy()
df_stock["Material"] = df_stock["Material"].astype(str).str.strip()

# Dédoublonnage (première occurrence conservée)
doublons = df_stock[df_stock["Material"].duplicated()]["Material"].unique()
if len(doublons): print(f"⚠️  {len(doublons)} PN dupliqué(s) → première occurrence conservée")
df_stock = df_stock.drop_duplicates(subset="Material", keep="first").set_index("Material")

# Séparation par type SAP : ZROH=MP | ZHLB=SF | ZFRT=PF
I = df_stock[df_stock["Material Type"] == "ZROH"].index.tolist()  # Matières Premières
J = df_stock[df_stock["Material Type"] == "ZHLB"].index.tolist()  # Semi-Finis
L = df_stock[df_stock["Material Type"] == "ZFRT"].index.tolist()  # Produits Finis

# Dictionnaires prix et stocks initiaux
prix_MP       = df_stock.loc[I, "Prix Unitaire ($)"].to_dict()
stock_init_MP = df_stock.loc[I, "Stock Actuel (unités)"].to_dict()
prix_SF       = df_stock.loc[J, "Prix Unitaire ($)"].to_dict()
stock_init_SF = df_stock.loc[J, "Stock Actuel (unités)"].to_dict()
prix_PF       = df_stock.loc[L, "Prix Unitaire ($)"].to_dict()
stock_init_PF = df_stock.loc[L, "Stock Actuel (unités)"].to_dict()

# Dictionnaire de type par PN (utile pour la BOM)
type_par_pn = df_stock["Material Type"].to_dict()

# ── Vérification ──────────────────────────────────────────────────────────────
val_MP  = sum(prix_MP[i]  * stock_init_MP[i]  for i in I)
val_SF  = sum(prix_SF[j]  * stock_init_SF[j]  for j in J)
val_PF  = sum(prix_PF[l]  * stock_init_PF[l]  for l in L)
val_tot = val_MP + val_SF + val_PF

print("=" * 62)
print("  STOCK INITIAL")
print("=" * 62)
print(f"  MP  : {len(I):3d} références  →  {val_MP:>14,.0f}  $")
print(f"  SF  : {len(J):3d} références  →  {val_SF:>14,.0f}  $")
print(f"  PF  : {len(L):3d} références  →  {val_PF:>14,.0f}  $")
print(f"  {'─'*48}")
print(f"  TOTAL                   →  {val_tot:>14,.0f}  $")
print("=" * 62)


---
### 6.2 — Paramètres globaux (`Parametres_Modele.xlsx`)

Fichier contenant une colonne `Paramètre` et une colonne `Valeur`.

In [ ]:
df_params = pd.read_excel(f("Parametres_Modele.xlsx"), sheet_name="Parametres_Globaux")
df_params.columns = df_params.columns.str.strip()
params = df_params.set_index("Paramètre")["Valeur"].to_dict()

TARGET  = float(params["TARGET"])    # cible fiscale fin septembre ($)
tau     = float(params["tau"])       # taux de possession annuel
z_alpha = float(params["z_alpha"])   # coeff. niveau de service (1.65 = 95%)
M       = float(params["M_bigM"])    # big-M linéarisation
V_max   = float(params["V_max"])     # capacité stockage globale (m³)

print("=" * 62)
print("  PARAMÈTRES GLOBAUX")
print("=" * 62)
print(f"  TARGET  (cible fiscale)  : {TARGET:>14,.0f}  $")
print(f"  V_max   (capacité stock) : {V_max:>14,.0f}  m³")
print(f"  tau     (possession)     : {tau*100:>13.0f}  %/an")
print(f"  z_alpha (niveau service) : {z_alpha:>14.2f}  (α ≈ 95%)")
print(f"  M       (big-M)          : {M:>14,.0f}")
print(f"  Écart à réduire          : {val_tot - TARGET:>14,.0f}  $")
print("=" * 62)


---
### 6.3 — Lead Times (`LeadTime.xlsx`)

Colonnes attendues : `PN`, `LEAD TIME` (en jours).
Conversion : $LT_{mois} = \max(1,\, \lceil LT_{jours} / 30 \rceil)$.
Si un PN MP a plusieurs fournisseurs, je prends le **maximum** (scénario prudent).


In [ ]:
df_lt = pd.read_excel(f("LEAD TIME FOURNISSEUR AVEC PN.xlsx"))
df_lt.columns = df_lt.columns.str.strip()
df_lt["PN"] = df_lt["PN"].astype(str).str.strip()

# Un PN peut avoir plusieurs lignes (multi-fournisseurs) → max du lead time
lt_jours = df_lt.groupby("PN")["LEAD TIME"].max().to_dict()

LT = {}
for i in I:
    jours = lt_jours.get(str(i), 30)   # 30 jours par défaut si manquant
    LT[i] = max(1, math.ceil(jours / 30))

# Vérification
mp_manquants = [i for i in I if str(i) not in lt_jours]
if mp_manquants:
    print(f"⚠️  {len(mp_manquants)} MP sans lead time → 1 mois par défaut : {mp_manquants[:5]}")

print("✅  Lead Times chargés :")
df_lt_check = pd.DataFrame(
    [{"PN": i, "LT (jours)": lt_jours.get(str(i), "—"), "LT (mois modèle)": LT[i]} for i in I]
).set_index("PN")
display(df_lt_check)


---
### 6.4 — MOQ et MLS (`MOQ_MLS.xlsx`)

Colonnes attendues : `Material`, `Cur. Minimum lot size`.
- Si le PN est dans I (MP) → **MOQ** (quantité minimale de commande fournisseur)
- Si dans J (SF) ou L (PF) → **MLS** (taille de lot minimale de production)


In [ ]:
df_moq = pd.read_excel(f("PN MLS.xls"))
df_moq.columns = df_moq.columns.str.strip()
df_moq["Material"] = df_moq["Material"].astype(str).str.strip()
df_moq = df_moq.set_index("Material")["Cur. Minimum lot size"]

MOQ    = {}
MLS_SF = {}
MLS_PF = {}

for i in I:
    v = df_moq.get(str(i), None)
    MOQ[i] = float(v) if (v is not None and not pd.isna(v)) else 1.0

for j in J:
    v = df_moq.get(str(j), None)
    MLS_SF[j] = float(v) if (v is not None and not pd.isna(v)) else 1.0

for l in L:
    v = df_moq.get(str(l), None)
    MLS_PF[l] = float(v) if (v is not None and not pd.isna(v)) else 1.0

# Rapport
n_moq_manq = sum(1 for i in I if MOQ[i] == 1.0 and str(i) not in df_moq.index)
n_mls_manq = sum(1 for j in J if MLS_SF[j] == 1.0) + sum(1 for l in L if MLS_PF[l] == 1.0)
print(f"✅  MOQ chargées  : {len(MOQ)} MP  ({n_moq_manq} avec défaut=1)")
print(f"   MLS SF        : {len(MLS_SF)} SF  |  MLS PF : {len(MLS_PF)} PF")


---
### 6.5 — Demande client (`Demande.xlsx`)

Mon fichier contient une ligne par PN PF et une colonne par semaine au format `FY25-W36` … `FY25-W51`.
J'agrège par somme pour obtenir la demande mensuelle $D_{l,t}$ (4 semaines par mois).


In [ ]:
df_dem = pd.read_excel(f("DEMAND ANALYSSIS.xlsx"))
df_dem.columns = df_dem.columns.str.strip()
df_dem["PN"] = df_dem["PN"].astype(str).str.strip()
df_dem = df_dem[df_dem["PN"].isin([str(l) for l in L])].set_index("PN")

# Colonnes semaines reconnues
week_cols = [col for col in df_dem.columns if col in week_to_t]

if not week_cols:
    raise ValueError(
        f"Aucune colonne semaine reconnue (attendu : FY25-W36…FY25-W51).
"
        f"Colonnes disponibles : {list(df_dem.columns[:10])}"
    )

# Agrégation hebdo → mensuelle
demande = {}
for l in L:
    for t in T:
        cols_t = [w for w in week_cols if week_to_t[w] == t]
        if str(l) in df_dem.index and cols_t:
            demande[(l, t)] = int(df_dem.loc[str(l), cols_t].fillna(0).sum())
        else:
            demande[(l, t)] = 0

# Vérification
df_dem_agg = pd.DataFrame(
    {noms_periodes[t]: {l: demande[(l, t)] for l in L} for t in T}
)
df_dem_agg.index.name = "PN (PF)"
print(f"✅  Demande mensuelle agrégée ({len(week_cols)} semaines lues) :")
display(df_dem_agg)


---
### 6.6 — Pipeline SAP : WIP et achats en cours (`WIP Achats En Cours.xlsx`)

Ce fichier contient les flux **déjà engagés** avant le démarrage du modèle.
Je répartis chaque quantité dans la période $t$ correspondant à la date de livraison/fin d'OF prévue.

Colonnes attendues :
- Feuille `Achats_En_Cours_MP` : `PN`, `Qté Commandée`, `Mois Modèle (t)`
- Feuille `WIP_Semi_Finis`     : `PN SF`, `Qté En-Cours`, `Mois Modèle (t)`
- Feuille `WIP_Produits_Finis` : `PN PF`, `Qté En-Cours`, `Mois Modèle (t)`


In [ ]:
# Initialisation à zéro pour tous les PN et toutes les périodes
OC     = {i: {t: 0.0 for t in T} for i in I}
WIP_SF = {j: {t: 0.0 for t in T} for j in J}
WIP_PF = {l: {t: 0.0 for t in T} for l in L}

xl_wip = f("WIP W3.xlsx")

# ── Achats MP en cours ────────────────────────────────────────────────────────
df_oc = pd.read_excel(xl_wip, sheet_name="Achats_En_Cours_MP")
df_oc.columns = df_oc.columns.str.strip()
df_oc["PN"] = df_oc["PN"].astype(str).str.strip()
for _, row in df_oc.iterrows():
    pn = row["PN"]; t = int(row["Mois Modèle (t)"]); q = float(row.get("Qté Commandée", 0) or 0)
    if pn in OC and t in T: OC[pn][t] += q

# ── WIP Semi-Finis ────────────────────────────────────────────────────────────
df_wsf = pd.read_excel(xl_wip, sheet_name="WIP_Semi_Finis")
df_wsf.columns = df_wsf.columns.str.strip()
df_wsf["PN SF"] = df_wsf["PN SF"].astype(str).str.strip()
for _, row in df_wsf.iterrows():
    pn = row["PN SF"]; t = int(row["Mois Modèle (t)"]); q = float(row.get("Qté En-Cours", 0) or 0)
    if pn in WIP_SF and t in T: WIP_SF[pn][t] += q

# ── WIP Produits Finis ────────────────────────────────────────────────────────
df_wpf = pd.read_excel(xl_wip, sheet_name="WIP_Produits_Finis")
df_wpf.columns = df_wpf.columns.str.strip()
df_wpf["PN PF"] = df_wpf["PN PF"].astype(str).str.strip()
for _, row in df_wpf.iterrows():
    pn = row["PN PF"]; t = int(row["Mois Modèle (t)"]); q = float(row.get("Qté En-Cours", 0) or 0)
    if pn in WIP_PF and t in T: WIP_PF[pn][t] += q

# ── Vérification ─────────────────────────────────────────────────────────────
print("✅  Pipeline SAP chargé :")
print("\n  Achats MP en cours (OC) — total par période :")
display(pd.DataFrame(
    {noms_periodes[t]: {i: OC[i][t] for i in I} for t in T}
).rename_axis("PN"))

print("\n  WIP SF — total par période :")
display(pd.DataFrame(
    {noms_periodes[t]: {j: WIP_SF[j][t] for j in J} for t in T}
).rename_axis("PN"))

print("\n  WIP PF — total par période :")
display(pd.DataFrame(
    {noms_periodes[t]: {l: WIP_PF[l][t] for l in L} for t in T}
).rename_axis("PN"))


---
### 6.7 — Nomenclature BOM (`BOM_SAP.xlsx`)

Structure du fichier extrait de SAP :

| Colonne | Signification |
|---------|---------------|
| `Parent` | PN du produit parent (SF ou PF) |
| `Base quantity` | Quantité de référence de la nomenclature (ex : 1 000 000) |
| `Material / Doc` | PN du composant (MP ou SF) |
| `Quantity` | Quantité du composant pour fabriquer `Base quantity` unités du parent |

La **quantité unitaire** (pour fabriquer 1 unité du parent) est :
$$q_{unitaire} = \frac{\text{Quantity}}{\text{Base quantity}}$$

Le type du parent et du composant est déterminé en croisant avec le référentiel Stock (type\_par\_pn).

**Règle de routage :**
- Composant MP → Parent SF : $a_{ij}$ (BOM MP→SF, contrainte C1)
- Composant SF → Parent PF : $b_{jl}$ (BOM SF→PF, contrainte C2)
- Composant MP → Parent PF : ignoré dans ce modèle (simplification 2 niveaux)


In [ ]:
df_bom = pd.read_excel(f("BOM_SAP.xlsx"))
df_bom.columns = df_bom.columns.str.strip()

# Normalisation des colonnes clés
df_bom["Parent"]        = df_bom["Parent"].astype(str).str.strip()
df_bom["Material / Doc"]= df_bom["Material / Doc"].astype(str).str.strip()
df_bom["Base quantity"] = pd.to_numeric(df_bom["Base quantity"], errors="coerce").fillna(1)
df_bom["Quantity"]      = pd.to_numeric(df_bom["Quantity"],       errors="coerce").fillna(0)

# Quantité unitaire : qté composant pour fabriquer 1 unité du parent
df_bom["qty_unitaire"] = df_bom["Quantity"] / df_bom["Base quantity"]

# Initialisation des matrices BOM à zéro
a_bom = {i: {j: 0.0 for j in J} for i in I}   # MP → SF
b_bom = {j: {l: 0.0 for l in L} for j in J}   # SF → PF

ignores = 0
for _, row in df_bom.iterrows():
    parent    = row["Parent"]
    composant = row["Material / Doc"]
    qty       = row["qty_unitaire"]

    type_parent    = type_par_pn.get(parent,    None)
    type_composant = type_par_pn.get(composant, None)

    if type_composant == "ZROH" and type_parent == "ZHLB":
        # MP → SF  →  matrice a
        if composant in a_bom and parent in a_bom[composant]:
            a_bom[composant][parent] += qty

    elif type_composant == "ZHLB" and type_parent == "ZFRT":
        # SF → PF  →  matrice b
        if composant in b_bom and parent in b_bom[composant]:
            b_bom[composant][parent] += qty

    else:
        ignores += 1   # MP→PF direct ou lignes hors périmètre

# ── Vérification ─────────────────────────────────────────────────────────────
n_a = sum(1 for i in I for j in J if a_bom[i][j] > 0)
n_b = sum(1 for j in J for l in L if b_bom[j][l] > 0)

print(f"✅  BOM chargée : {len(df_bom)} lignes lues")
print(f"   Liens MP→SF (a≠0) : {n_a}  |  Liens SF→PF (b≠0) : {n_b}  |  Lignes ignorées : {ignores}")

print("\n  Matrice a (MP→SF) — quantité unitaire :")
df_a = pd.DataFrame(a_bom).T.loc[I, J] if (I and J) else pd.DataFrame()
display(df_a)

print("\n  Matrice b (SF→PF) — quantité unitaire :")
df_b = pd.DataFrame(b_bom).T.loc[J, L] if (J and L) else pd.DataFrame()
display(df_b)


---
### 6.8 — Application des données générées logiquement

Appel des fonctions définies en section 5 maintenant que les listes I / J / L et les prix sont connus.

In [ ]:
# Volumes unitaires de stockage
volume_MP, volume_SF, volume_PF = generer_volumes(prix_MP, prix_SF, prix_PF)

# Charges, temps de cycle
charge_SF, charge_PF, ct_SF, ct_PF = generer_charges_et_cycles(prix_SF, prix_PF)

# Capacités de production
Cap_SF, Cap_PF = generer_capacites(T, t_aout)

# Coûts fixes (pour EOQ)
C_passation, C_lancement_SF, C_lancement_PF = generer_couts_fixes(prix_MP, prix_SF, prix_PF)

# Écart-types de la demande
sigma_D = generer_sigma_D(stock_init_MP, stock_init_SF, stock_init_PF, demande, T)

# ── Affichage de synthèse ─────────────────────────────────────────────────────
print("=" * 65)
print("  DONNÉES GÉNÉRÉES LOGIQUEMENT")
print("=" * 65)
rows = []
for i in I:
    rows.append({"PN":i,"Type":"MP","Vol (m³/u)":volume_MP[i],
                 "Charge (h/u)":"—","Cycle (mois)":"—","σ_D":sigma_D[i]})
for j in J:
    rows.append({"PN":j,"Type":"SF","Vol (m³/u)":volume_SF[j],
                 "Charge (h/u)":charge_SF[j],"Cycle (mois)":ct_SF[j],"σ_D":sigma_D[j]})
for l in L:
    rows.append({"PN":l,"Type":"PF","Vol (m³/u)":volume_PF[l],
                 "Charge (h/u)":charge_PF[l],"Cycle (mois)":ct_PF[l],"σ_D":sigma_D[l]})
display(pd.DataFrame(rows).set_index("PN"))

print("\n  Capacités de production (heures/mois) :")
df_cap = pd.DataFrame({"Cap SF (h)":Cap_SF,"Cap PF (h)":Cap_PF})
df_cap.index = [noms_periodes[t] for t in T]
display(df_cap)


---
### 6.9 — Calibration EOQ — Stocks de sécurité $SS_k$

$$Q_k^* = \sqrt{\frac{2\,D_k\,C_k}{h_k}}, \quad h_k = \tau \cdot p_k, \quad SS_k = z_\alpha \cdot \sigma_{D_k} \cdot \sqrt{LT_k \text{ ou } ct_k}$$

La demande annuelle $D_k$ est approchée par cascade BOM : demande PF réelle × coefficients BOM.


In [ ]:
# Demande annuelle approchée par cascade BOM (PF → SF → MP)
D_annuel_PF = {l: sum(demande[(l, t)] for t in T) * 3 for l in L}  # ×3 → annualisé

# SF : somme pondérée des demandes PF par les coefficients BOM b[j][l]
D_annuel_SF = {}
for j in J:
    d = sum(b_bom[j][l] * D_annuel_PF[l] for l in L)
    D_annuel_SF[j] = max(1, d)

# MP : somme pondérée des demandes SF par les coefficients BOM a[i][j]
D_annuel_MP_calc = {}
for i in I:
    d = sum(a_bom[i][j] * D_annuel_SF[j] for j in J)
    D_annuel_MP_calc[i] = max(1, d)

SS = {}
resultats_eoq = []

for i in I:
    h  = tau * prix_MP[i]
    Qs = math.sqrt(2 * D_annuel_MP_calc[i] * C_passation[i] / h)
    ss = z_alpha * sigma_D[i] * math.sqrt(LT[i])
    SS[i] = ss
    resultats_eoq.append({"PN":i,"Étage":"MP","h($/an)":round(h,2),
                          "EOQ*(unités)":round(Qs),"SS(unités)":round(ss)})

for j in J:
    h  = tau * prix_SF[j]
    Qs = math.sqrt(2 * D_annuel_SF[j] * C_lancement_SF[j] / h)
    ss = z_alpha * sigma_D[j] * math.sqrt(ct_SF[j])
    SS[j] = ss
    resultats_eoq.append({"PN":j,"Étage":"SF","h($/an)":round(h,2),
                          "EOQ*(unités)":round(Qs),"SS(unités)":round(ss)})

for l in L:
    h  = tau * prix_PF[l]
    Qs = math.sqrt(2 * D_annuel_PF[l] * C_lancement_PF[l] / h)
    ss = z_alpha * sigma_D[l] * math.sqrt(ct_PF[l])
    SS[l] = ss
    resultats_eoq.append({"PN":l,"Étage":"PF","h($/an)":round(h,2),
                          "EOQ*(unités)":round(Qs),"SS(unités)":round(ss)})

SS_MP = {i: SS[i] for i in I}
SS_SF = {j: SS[j] for j in J}
SS_PF = {l: SS[l] for l in L}

print("✅  Calibration EOQ — Stocks de sécurité calculés :")
display(pd.DataFrame(resultats_eoq).set_index("PN"))


---
### 6.10 — Construction du `DataPortal` et création de l'instance

Tous les paramètres lus et générés sont injectés dans le modèle abstrait Pyomo.

In [ ]:
data = DataPortal()

# Horizon et ensembles
data['T'] = T_max;  data['t_juillet'] = t_juillet;  data['AoutSet'] = t_aout
data['MP'] = I;     data['SF'] = J;                  data['PF'] = L

# Paramètres globaux
data['TARGET'] = TARGET;  data['tau'] = tau;  data['z_alpha'] = z_alpha
data['M'] = M;            data['V_max'] = V_max

# MP
data['prix_MP']       = prix_MP
data['stock_init_MP'] = stock_init_MP
data['LT']            = LT
data['MOQ']           = MOQ
data['volume_MP']     = volume_MP
data['SS_MP']         = SS_MP

# SF
data['prix_SF']       = prix_SF
data['stock_init_SF'] = stock_init_SF
data['ct_SF']         = ct_SF
data['MLS_SF']        = MLS_SF
data['charge_SF']     = charge_SF
data['volume_SF']     = volume_SF
data['SS_SF']         = SS_SF

# PF
data['prix_PF']       = prix_PF
data['stock_init_PF'] = stock_init_PF
data['ct_PF']         = ct_PF
data['MLS_PF']        = MLS_PF
data['charge_PF']     = charge_PF
data['volume_PF']     = volume_PF
data['SS_PF']         = SS_PF

# Flux
data['demande'] = demande
data['OC']      = {(i, t): OC[i][t]      for i in I for t in T}
data['WIP_SF']  = {(j, t): WIP_SF[j][t]  for j in J for t in T}
data['WIP_PF']  = {(l, t): WIP_PF[l][t]  for l in L for t in T}
data['a']       = {(i, j): a_bom[i][j]   for i in I for j in J if a_bom[i][j] > 0}
data['b']       = {(j, l): b_bom[j][l]   for j in J for l in L if b_bom[j][l] > 0}
data['Cap_SF']  = Cap_SF
data['Cap_PF']  = Cap_PF

# Instance concrète
instance = model.create_instance(data)

print("=" * 62)
print("  ✅  INSTANCE CONCRÈTE — PRÊTE À RÉSOUDRE")
print("=" * 62)
print(f"  MP  : {len(I)} références  |  SF : {len(J)}  |  PF : {len(L)}")
print(f"  Périodes        : {[noms_periodes[t] for t in T]}")
print(f"  Valeur initiale : {val_tot:>14,.0f}  $")
print(f"  Cible TARGET    : {TARGET:>14,.0f}  $")
print(f"  Écart à réduire : {val_tot - TARGET:>14,.0f}  $")
print(f"  V_max           : {V_max:>14,.0f}  m³")
print(f"  Contraintes     : {len(instance.constraints)}")
print("=" * 62)
print("\n  ▶  Lance : resoudre(instance) pour obtenir la solution optimale")


---
### 5.4 Résolution (appel réel de `resoudre`, défini à l'étape 4)

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=False)

print("=" * 55)
print("  STATUT DE LA RÉSOLUTION")
print("=" * 55)
print(f"  Statut solveur      : {resultats.solver.termination_condition}")
print(f"  Valeur objectif (Z) : {value(instance.cost):>14,.2f} $")
print("=" * 55)


---
### 5.5 Vérification de la cible fiscale (C5)

In [ ]:
T_max = value(instance.T)

# Valeur totale du stock en fin d'horizon (fin septembre)
valeur_fin_sept = (
    sum(instance.prix_MP[i] * value(instance.S_MP[i, T_max]) for i in instance.MP)
  + sum(instance.prix_SF[j] * value(instance.S_SF[j, T_max]) for j in instance.SF)
  + sum(instance.prix_PF[l] * value(instance.S_PF[l, T_max]) for l in instance.PF)
)

print(f"  Valeur stock fin {noms_periodes[T_max]:<9}: {valeur_fin_sept:>14,.0f} $")
print(f"  Cible fiscale (TARGET)        : {value(instance.TARGET):>14,.0f} $")
print()
if valeur_fin_sept <= value(instance.TARGET):
    print("  ✅ Cible atteinte !")
else:
    ecart = valeur_fin_sept - value(instance.TARGET)
    print(f"  ❌ Cible dépassée — Écart : {ecart:,.0f} $")


---
### 5.6 Extraction des résultats en DataFrames

In [ ]:
def extraire(var, refs, label):
    """Extrait une variable Pyomo en DataFrame (références x périodes)."""
    lignes = [
        {"Référence": r, "Période": noms_periodes[t], "t": t, label: value(var[r, t])}
        for r in refs for t in instance.PER
    ]
    df = pd.DataFrame(lignes)
    display(df.pivot(index="Référence", columns="Période", values=label)[
        [noms_periodes[t] for t in instance.PER]
    ])
    return df

print("Stocks MP  :") ; df_S_MP = extraire(instance.S_MP, instance.MP, "Stock MP")
print("Stocks SF  :") ; df_S_SF = extraire(instance.S_SF, instance.SF, "Stock SF")
print("Stocks PF  :") ; df_S_PF = extraire(instance.S_PF, instance.PF, "Stock PF")
print("Commandes O_MP  :") ; df_O_MP = extraire(instance.O_MP, instance.MP, "O_MP")
print("Production P_SF :") ; df_P_SF = extraire(instance.P_SF, instance.SF, "P_SF")
print("Production P_PF :") ; df_P_PF = extraire(instance.P_PF, instance.PF, "P_PF")


---
### 5.7 Valeur de stock par période et graphiques

In [ ]:
# ── Tableau de valeur de stock par période et par étage ───────────────────────
valeurs = []
for t in instance.PER:
    v_mp = sum(instance.prix_MP[i] * value(instance.S_MP[i, t]) for i in instance.MP)
    v_sf = sum(instance.prix_SF[j] * value(instance.S_SF[j, t]) for j in instance.SF)
    v_pf = sum(instance.prix_PF[l] * value(instance.S_PF[l, t]) for l in instance.PF)
    valeurs.append({
        "Période": noms_periodes[t],
        "MP ($)": round(v_mp), "SF ($)": round(v_sf), "PF ($)": round(v_pf),
        "Total ($)": round(v_mp + v_sf + v_pf)
    })
df_valeur = pd.DataFrame(valeurs).set_index("Période")

print("Valeur de stock par période et par étage :")
display(df_valeur)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

# Courbe par étage
for col, color in [("MP ($)", "steelblue"), ("SF ($)", "darkorange"), ("PF ($)", "seagreen")]:
    ax.plot(df_valeur.index, df_valeur[col], marker="o", label=col, color=color)

# Courbe Total
ax.plot(df_valeur.index, df_valeur["Total ($)"],
        marker="s", linewidth=2.5, color="black", label="Total")

# Ligne cible fiscale
ax.axhline(value(instance.TARGET), color="red", linestyle="--", linewidth=1.5,
           label=f"Cible fiscale ({value(instance.TARGET):,.0f} $)")

# Annotation : valeur initiale
ax.scatter([df_valeur.index[0]], [val_total_init], color="black", zorder=5, s=80)
ax.annotate(f"Valeur initiale\n{val_total_init:,.0f} $",
            xy=(df_valeur.index[0], val_total_init),
            xytext=(0.1, 0.95), textcoords="axes fraction",
            fontsize=9, color="black")

ax.set_title("Trajectoire de la valeur de stock par étage — Juin à Septembre",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Période")
ax.set_ylabel("Valeur du stock ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---
### 5.8 Résumé final

In [ ]:
print("=" * 60)
print("  RÉSUMÉ FINAL DE LA RÉSOLUTION")
print("=" * 60)
print(f"  Statut solveur          : {resultats.solver.termination_condition}")
print(f"  Valeur objectif Z       : {value(instance.cost):>14,.0f} $")
print(f"  Valeur stock fin {noms_periodes[T_max]:<9}: {valeur_fin_sept:>14,.0f} $")
print(f"  Cible fiscale (TARGET)  : {value(instance.TARGET):>14,.0f} $")
print(f"  Capacité stockage V_max : {value(instance.V_max):>14,.0f} m³")
print(f"  Écart cible             : {valeur_fin_sept - value(instance.TARGET):>+14,.0f} $")
print(f"  Cible respectée         : {'✅ Oui' if valeur_fin_sept <= value(instance.TARGET) else '❌ Non'}")
print("=" * 60)
